In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

In [ ]:


# 1. Define the directory path
folder_path = Path("../dataset/dsfsi-anv/anv")

# 2. Get all CSV files in the folder (use rglob("*.csv") to include subfolders)
csv_files = list(folder_path.glob("*.csv"))

In [ ]:
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
df = df.astype({"language":'category',"split":'category','audio_id':'string','recorder_uuid':'string'},)
df = df.astype({'type':'category','domain':'category','topic':'category',})
df = df.astype({'scenario':'string','transcript':'string','document_id':'string','source_document':'category'})
df = df.astype({'microphone_device_id':'string','microphone_label':'category'})

In [ ]:
df.info(show_counts=True)

In [ ]:
df.isnull().sum()

In [ ]:
drop_coloumns = ['system_file_name','file_name','full_path']
display(df.drop(columns=drop_coloumns).describe(include='all'))
#drop_coloums = ['langauge','system_file_name','file_name','full_path']

In [ ]:
# Use display() combined with Markdown()
display(Markdown(f"## Now viewing: **Language**"))

plt.figure(figsize=(6, 4))
df['language'].value_counts().plot(kind="bar")
plt.title(f"Value Counts for Column: language")
plt.xlabel("Language")
plt.ylabel("Count")
plt.show()

In [ ]:
languages = df['language'].unique()
languages

In [ ]:
# lang_dict = {
#     "isiNdebele": "NBL",
#     "isiXhosa": "XHO",
#     "isiZulu": "ZUL",
#     "Sesotho": "SOT",
#     "Setswana": "TSN",
#     "Tshivenda": "VEN",
#     "Xitsonga": "TSO",
# }

lang_dict = {
    "Setswana": "TSN",
    "Tshivenda": "VEN",
}

In [ ]:
N = 2 #number of languages to display

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]
    display_coloumns = ['split','type','domain','microphone_label', 'source_document']
    for col in df_filtered[display_coloumns].columns:
        # Use display() combined with Markdown()
        display(Markdown(f"## Now viewing {language} ({code}): **{col}**"))

        plt.figure(figsize=(6, 4))
        df_filtered[col].value_counts().plot(kind="bar")
        plt.title(f"Value Counts for {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]
    display_coloumns = ['duration','size_bytes','signal_to_noise_ratio','audio_num_samples']
    for col in df_filtered[display_coloumns].columns:

        display(Markdown(f"## Now viewing: **{col}**"))

        # Pass the data array/series directly
        plt.hist(df_filtered[col], bins=15, color="skyblue", edgecolor="black")

        # Adding labels
        plt.title(f"Distribution of {col}")
        plt.xlabel(f"{col} Group")
        plt.ylabel("Count")

        # Display the plot
        plt.show()

        display(df_filtered[col].describe())

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    df_filtered = df[df['language'] == code]

    display(Markdown(f"## Now viewing: **Data quality checks for {language} ({code})**"))

    display(Markdown(f"### Sampling rates: {df_filtered['audio_sampling_rate'].value_counts().to_dict()}"))
    display(Markdown(f"### Channels: {df_filtered['audio_num_channels'].value_counts().to_dict()}"))

    duplicate_audio_ids = df_filtered['audio_id'].duplicated().sum()
    display(Markdown(f"### Number of duplicate audio_ids: {duplicate_audio_ids}"))

    clips_under_1s = (df_filtered['duration'] < 1).sum()
    clips_under_5s = (df_filtered['duration'] < 5).sum()
    clips_over_30s = (df_filtered['duration'] > 30).sum()
    display(Markdown(f"### Clips < 1s: {clips_under_1s} | Clips < 5s: {clips_under_5s} | Clips > 30s: {clips_over_30s}"))

    snr_below_15 = (df_filtered['signal_to_noise_ratio'] < 15).sum()
    min_snr = df_filtered['signal_to_noise_ratio'].min()
    display(Markdown(f"### Clips with SNR < 15 dB: {snr_below_15} (min SNR = {min_snr:.1f} dB)"))

    recorders_in_multiple_splits = (df_filtered.groupby('recorder_uuid')['split'].nunique() > 1).sum()
    display(Markdown(f"### Recorders appearing in more than one split: {recorders_in_multiple_splits}"))

# Transcript Analysis

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]

    display(Markdown(f"## Now viewing: **Top words in {language} ({code})**"))

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_colwidth", None)

    for category in sorted(df_filtered['type'].dropna().unique()):
        subset = df_filtered[df_filtered['type'] == category][['type', 'transcript']].head(10)
        display(Markdown(f"### Type: {category}"))
        display(subset)

    #count none or empty transcripts
    empty_transcripts = df_filtered['transcript'].isnull().sum() + (df_filtered['transcript'].str.strip() == '').sum()
    display(Markdown(f"### Number of empty or null transcripts: {empty_transcripts}"))


    df_filtered['transcript_word_count'] = df_filtered['transcript'].apply(lambda x: len(str(x).split()))
    plt.hist(df_filtered['transcript_word_count'], bins=20, color="lightgreen", edgecolor="black")
    plt.title("Distribution of Transcript Word Counts")
    plt.xlabel("Word Count")
    plt.ylabel("Frequency")
    plt.show()


    display(Markdown(f"## Now viewing: **Top words in transcript**"))
    from collections import Counter
    all_words = ' '.join(df_filtered['transcript'].dropna()).lower().split()
    word_counts = Counter(all_words)
    top_words = word_counts.most_common(20)
    display(Markdown(f"### Top Words in Transcript"))
    for i, (word, count) in enumerate(top_words, start=1):
        display(Markdown(f"{i}. **{word}**: {count}"))

In [ ]:
for language, code in list(lang_dict.items())[:N]:
    df_filtered = df[df['language'] == code]

    df_filtered = df_filtered.copy()
    df_filtered['transcript_char_count'] = df_filtered['transcript'].apply(lambda x: len(str(x)))

    plt.hist(df_filtered['transcript_char_count'], bins=20, color="lightgreen", edgecolor="black")
    plt.title(f"Distribution of Transcript Character Counts for {language} ({code})")
    plt.xlabel("Character Count")
    plt.ylabel("Frequency")
    plt.show()

    display(Markdown(f"## Now viewing: **Top characters in transcript for {language} ({code})**"))
    from collections import Counter
    all_chars = ''.join(df_filtered['transcript'].dropna()).lower()
    filtered_chars = [c for c in all_chars if not c.isspace()]
    char_counts = Counter(filtered_chars)
    top_chars = char_counts.most_common(20)

    number_of_unique_chars = len(char_counts)
    display(Markdown(f"### Number of Unique Characters in Transcript: {number_of_unique_chars}"))


    display(Markdown(f"### Top 20 Characters in Transcript"))
    for i, (char, count) in enumerate(top_chars, start=1):
        display(Markdown(f"{i}. **{char}**: {count}"))

# NCHLT Analysis (Tshivenda)

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
nchlt_df = pd.read_csv("../dataset/dsfsi-anv/nchlt/multilingual-nchlt-dataset_ven.csv")
nchlt_df.shape

In [ ]:
nchlt_df.head(5)

In [ ]:
nchlt_df.isnull().sum()

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    nchlt_filtered = nchlt_df[nchlt_df['language_code'] == code.lower()]
    display_coloumns = ['split','gender','audio_sampling_rate','audio_num_channels']
    for col in nchlt_filtered[display_coloumns].columns:
        # Use display() combined with Markdown()
        display(Markdown(f"## Now viewing {language} ({code}): **{col}**"))

        plt.figure(figsize=(6, 4))
        nchlt_filtered[col].value_counts().plot(kind="bar")
        plt.title(f"Value Counts for {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    nchlt_filtered = nchlt_df[nchlt_df['language_code'] == code.lower()]
    display_coloumns = ['audio_duration_s','audio_size_bytes','audio_num_samples']
    for col in nchlt_filtered[display_coloumns].columns:

        display(Markdown(f"## Now viewing: **{col}**"))

        # Pass the data array/series directly
        plt.hist(nchlt_filtered[col], bins=15, color="skyblue", edgecolor="black")

        # Adding labels
        plt.title(f"Distribution of {col}")
        plt.xlabel(f"{col} Group")
        plt.ylabel("Count")

        # Display the plot
        plt.show()

        display(nchlt_filtered[col].describe())

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    nchlt_filtered = nchlt_df[nchlt_df['language_code'] == code.lower()]

    display(Markdown(f"## Now viewing: **Speaker balance for {language} ({code})**"))

    number_of_unique_speakers = nchlt_filtered['speaker'].nunique()
    display(Markdown(f"### Number of Unique Speakers: {number_of_unique_speakers}"))

    plt.hist(nchlt_filtered['speaker'].value_counts(), bins=15, color="skyblue", edgecolor="black")
    plt.title(f"Distribution of Clips per Speaker for {language} ({code})")
    plt.xlabel("Clips per Speaker")
    plt.ylabel("Number of Speakers")
    plt.show()

    display(nchlt_filtered['speaker'].value_counts().describe())

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    nchlt_filtered = nchlt_df[nchlt_df['language_code'] == code.lower()]

    display(Markdown(f"## Now viewing: **Text checks for {language} ({code})**"))

    #count none or empty transcripts
    empty_transcripts = nchlt_filtered['text'].isnull().sum() + (nchlt_filtered['text'].str.strip() == '').sum()
    display(Markdown(f"### Number of empty or null transcripts: {empty_transcripts}"))

    #count duplicate filenames
    duplicate_filenames = nchlt_filtered['filename'].duplicated().sum()
    display(Markdown(f"### Number of duplicate filenames: {duplicate_filenames}"))

    nchlt_filtered = nchlt_filtered.copy()
    nchlt_filtered['text_word_count'] = nchlt_filtered['text'].apply(lambda x: len(str(x).split()))
    plt.hist(nchlt_filtered['text_word_count'], bins=20, color="lightgreen", edgecolor="black")
    plt.title(f"Distribution of Text Word Counts for {language} ({code})")
    plt.xlabel("Word Count")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Tshivenda only - Setswana/Sepedi are handled by their own sub-studies
for language, code in [("Tshivenda", "VEN")]:
    nchlt_filtered = nchlt_df[nchlt_df['language_code'] == code.lower()]

    display(Markdown(f"## Now viewing: **Data quality checks for {language} ({code})**"))

    clips_under_1s = (nchlt_filtered['audio_duration_s'] < 1).sum()
    clips_under_5s = (nchlt_filtered['audio_duration_s'] < 5).sum()
    clips_over_30s = (nchlt_filtered['audio_duration_s'] > 30).sum()
    display(Markdown(f"### Clips < 1s: {clips_under_1s} | Clips < 5s: {clips_under_5s} | Clips > 30s: {clips_over_30s}"))

    speakers_in_multiple_splits = (nchlt_filtered.groupby('speaker')['split'].nunique() > 1).sum()
    display(Markdown(f"### Speakers appearing in more than one split: {speakers_in_multiple_splits}"))

    all_chars = sorted(set(''.join(nchlt_filtered['text'].dropna().astype(str)).lower()))
    display(Markdown(f"### Number of Unique Characters in Text: {len(all_chars)}"))
    display(Markdown(f"### Character set: `{''.join(all_chars)}`"))

# Tshivenda EDA Findings & Filtering Rules

**NCHLT (ven): 49,748 clips, 56.3 hrs**
- Already 16 kHz mono throughout - no resampling needed.
- Splits: 30,984 train / 10,617 validation / 8,147 test. No speaker appears in more than one split (no leakage).
- 0 empty/null transcripts, 0 duplicate filenames. 295 duplicate (speaker, text) pairs = same prompt re-read, harmless.
- 208 speakers, median 237 clips each; gender skews male (30,781 vs 18,967) - worth reporting, not fixable.
- Durations: median 3.7 s, max 24.5 s, only 4 clips < 1 s. No VAD segmentation needed (nothing exceeds 30 s).
- Character set is clean: a-z, space, plus Tshivenda diacritics ḓ ḽ ṅ ṋ ṱ (32 chars total).

**ANV / Swivuriso (ven): 40,359 clips, 238.4 hrs**
- All 48 kHz mono - **must resample to 16 kHz** for Wav2Vec2/Whisper.
- Splits: 36,344 train / 2,360 dev / 1,655 dev_test. No recorder appears in more than one split.
- 0 empty/null transcripts, 0 duplicate audio_ids.
- Durations: median 17.7 s, max 180.7 s. **8,028 clips (20%) exceed 30 s** - these need VAD segmentation into 5-30 s windows. 1,171 clips < 5 s (keep; still usable). 0 clips < 1 s.
- SNR: minimum is 30 dB - nothing is noisy enough to drop. No SNR filter needed.
- Character set (54 chars) includes digits 0-9, punctuation (! " ' , - . ?), brackets, backslash, and a non-breaking space (\xa0) - **transcripts need normalisation before CTC tokenizer building**.

**Filtering / preprocessing rules to carry forward:**
1. NCHLT: drop the 4 clips < 1 s; use as-is otherwise (already 16 kHz mono, already short clips).
2. ANV: resample 48 kHz to 16 kHz; VAD-segment the 8,028 clips > 30 s into 5-30 s windows; drop nothing on SNR or transcripts.
3. Text normalisation (both corpora): lowercase; strip punctuation and digits. Note: the fine-tuning notebook's CHARS_TO_IGNORE regex does not yet cover backslash or the non-breaking space (\xa0) found in ANV transcripts - extend it.
4. CTC tokenizer vocabulary must include the five Tshivenda diacritic characters: ḓ ḽ ṅ ṋ ṱ.
